In [4]:
import pandas as pd
import numpy as np
import glob
import os
import plotly.express as px

# Load VN30 index from local file
vn30_path = 'data/coveraties/VN30.csv'
vn30 = pd.read_csv(vn30_path, parse_dates=['Date'], index_col='Date')['Close']
vn30.name = 'VN30'

# Khởi tạo DataFrame chứa giá Close
close_prices = pd.DataFrame()
close_prices['VN30'] = vn30

# Lấy các mã trong thư mục selected
selected_dir = 'data/raw/selected'
csv_files = glob.glob(os.path.join(selected_dir, "*.csv"))

for file in csv_files:
    ticker = os.path.basename(file).split('.')[0].upper()
    stock_df = pd.read_csv(file, parse_dates=['Date'], index_col='Date')
    
    if not stock_df.empty and 'Close' in stock_df.columns:
        close_prices[ticker] = stock_df['Close']

# Tính returns log cho tất cả các mã (bao gồm cả VN30)
returns = np.log(close_prices / close_prices.shift(1)).dropna()

# Tính ma trận tương quan (Correlation Matrix)
corr_matrix = returns.corr()

# Trực quan hóa bằng Heatmap (Biểu đồ nhiệt)
fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r", # Thang màu từ Đỏ (âm) sang Trắng (0) sang Xanh (dương)
    title="Ma trận tương quan (Correlation Matrix) giữa các cổ phiếu và VN30",
    zmin=-1, zmax=1
)

fig.update_layout(
    width=900,
    height=800,
    font=dict(size=12, family="Arial")
)

fig.show()

In [8]:
import pandas as pd
import numpy as np
import glob
import os
from scipy import stats
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Load VN30 index
vn30_path = 'data/coveraties/VN30.csv'
vn30      = pd.read_csv(
    vn30_path, parse_dates=['Date'], index_col='Date'
)['Close']
vn30_ret  = np.log(vn30 / vn30.shift(1)).dropna()

selected_dir = 'data/raw/selected'
csv_files    = glob.glob(
    os.path.join(selected_dir, "*.csv")
)

all_returns = {}  # Dùng để tính portfolio tracking
results     = []

for file in sorted(csv_files):
    ticker   = os.path.basename(file).split('.')[0].upper()
    stock_df = pd.read_csv(
        file, parse_dates=['Date'], index_col='Date'
    )
    
    if stock_df.empty or 'Close' not in stock_df.columns:
        continue
    
    ret    = np.log(
        stock_df['Close'] / stock_df['Close'].shift(1)
    ).dropna()
    common = ret.index.intersection(vn30_ret.index)
    
    if len(common) < 200:
        continue
    
    r_stock = ret[common]
    r_index = vn30_ret[common]
    
    # Tính p-value để chứng minh significant
    corr, pval = stats.pearsonr(r_stock, r_index)
    
    sig = ('***' if pval < 0.001 else
           '**'  if pval < 0.01  else
           '*'   if pval < 0.05  else '(ns)')
    
    results.append({
        'Ticker': ticker,
        'Corr':   corr,
        'PValue': pval,
        'N_days': len(common),
        'Sig':    sig
    })
    
    # Lưu lại để tính portfolio
    all_returns[ticker] = r_stock

# Chuyển results thành DataFrame và sắp xếp
df_results = pd.DataFrame(results).sort_values('Corr', ascending=True)

# Tính Portfolio tracking correlation
if len(all_returns) > 1:
    ret_df       = pd.DataFrame(all_returns)
    common_dates = ret_df.dropna(
        thresh=int(len(all_returns) * 0.8)
    ).index.intersection(vn30_ret.index)
    
    port_ret = ret_df.loc[common_dates].mean(axis=1)
    idx_ret  = vn30_ret[common_dates]
    
    tracking_corr, tracking_pval = stats.pearsonr(
        port_ret, idx_ret
    )
    r_squared = tracking_corr ** 2

# Xây dựng Plotly Chart hiển thị danh sách Corr và P-Value
fig = go.Figure()

# Thêm Bar Chart cho hệ số Correlation
fig.add_trace(go.Bar(
    x=df_results['Corr'],
    y=df_results['Ticker'],
    orientation='h',
    marker=dict(
        color=df_results['Corr'],
        colorscale='Blues'
    ),
    text=[f"{c:.3f} ({s})" for c, s in zip(df_results['Corr'], df_results['Sig'])],
    textposition='outside',
    name='Correlation'
))

# Cập nhật Layout
fig.update_layout(
    title=f"Tương quan của từng mã với VN30 (Tracking Corr: {tracking_corr:.3f}***, R²: {r_squared:.3f})",
    xaxis_title="Hệ số Tương quan (Pearson)",
    yaxis_title="Mã Cổ phiếu",
    height=600,
    font=dict(size=13, family="Arial")
)

# Thêm giới hạn trục để hiển thị text rõ nét
fig.update_xaxes(range=[0, df_results['Corr'].max() * 1.15])

# Hiển thị biểu đồ
fig.show()

# Save kết quả để đưa vào paper
df_results.to_csv(
    'data/representativeness.csv', index=False
)
print(f"Saved: data/representativeness.csv")
print(f"Portfolio Tracking Corr: {tracking_corr:.3f}")
print(f"Giải thích {r_squared*100:.1f}% variance của VN30")

Saved: data/representativeness.csv
Portfolio Tracking Corr: 0.964
Giải thích 93.0% variance của VN30
